In [1]:
from pathlib import Path
import sys

import pandas as pd

HERE = Path.cwd().resolve()

PYTHON_ROOT = next(
    path
    for path in (HERE, *HERE.parents)
    if (path / "trading_portfolio").is_dir()
    and (path / "t212_universe").is_dir()
)

PROJECT_DIR = (
    PYTHON_ROOT
    / "trading_portfolio"
    / "uk_portfolio"
    / "equity_momentum"
)

UNIVERSE_DIR = PYTHON_ROOT / "t212_universe"

source_path = str(PROJECT_DIR / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

BASELINE = {
    "formation_sessions": 252,
    "skip_sessions": 21,
    "top_frac": 0.20,
    "rebalance_frequency": "monthly",
    "initial_capital_gbp": 10_000.0,
}

display(pd.Series(BASELINE, name="Baseline").to_frame())

,Baseline
formation_sessions,252
skip_sessions,21
top_frac,0.2
rebalance_frequency,monthly
initial_capital_gbp,10000.0


In [2]:
import importlib
import momentum_data

importlib.reload(momentum_data)

universe = momentum_data.load_candidate_universe(
    UNIVERSE_DIR,
    PROJECT_DIR / "data" / "candidate_universe_v1.csv",
)

print(f"Frozen share candidates: {len(universe):,}")

display(
    universe[
        ["yf_symbol", "name", "isin", "quote_unit", "exchange"]
    ].head(10)
)

Frozen share candidates: 669


,yf_symbol,name,isin,quote_unit,exchange
0,88E.L,88 Energy,AU00000088E2,GBX,London Stock Exchange AIM
1,AURA.L,Aura Energy,AU000000AEE7,GBX,London Stock Exchange AIM
2,CLA.L,Celsius Resources,AU000000CLA6,GBX,London Stock Exchange AIM
3,EMH.L,European Metals Holdings,AU000000EMH5,GBX,London Stock Exchange AIM
4,GEO.L,Geo Exploration,AU000000GBP6,GBX,London Stock Exchange AIM
5,LIT.L,Litigation Capital Management,AU000000LCA6,GBX,London Stock Exchange AIM
6,SVML.L,Sovereign Metals,AU000000SVM6,GBX,London Stock Exchange AIM
7,WNX.L,Wellnex Life,AU0000162281,GBX,London Stock Exchange AIM
8,SYN.L,Synergia Energy,AU0000233538,GBX,London Stock Exchange AIM
9,ALL.L,Atlantic Lithium,AU0000237554,GBX,London Stock Exchange AIM


In [3]:
importlib.reload(momentum_data)

CACHE_PATH = (
    PROJECT_DIR.parent
    / "reversal_signals"
    / "data"
    / "uk_extension_2023_2025_v1"
    / "prepared_2015_2025_v1.pkl"
)

cached_data = momentum_data.load_cached_prices(CACHE_PATH, universe)

cached_prices = cached_data["prices"]
schedule = cached_data["schedule"]
price_coverage = cached_data["coverage"]

display(
    price_coverage["has_cached_prices"]
    .value_counts()
    .rename_axis("Has cached prices")
    .to_frame("Candidates")
)

display(price_coverage.head(10))

,Candidates
Has cached prices,
True,573
False,96


,yf_symbol,isin,name,cached_close_sessions,first_cached_session,last_cached_session,has_cached_prices
0,88E.L,AU00000088E2,88 Energy,2778,2015-01-02,2025-12-31,True
1,AURA.L,AU000000AEE7,Aura Energy,2347,2016-09-12,2025-12-31,True
2,CLA.L,AU000000CLA6,Celsius Resources,739,2023-01-30,2025-12-31,True
3,EMH.L,AU000000EMH5,European Metals Holdings,2540,2015-12-10,2025-12-31,True
4,GEO.L,AU000000GBP6,Geo Exploration,2779,2015-01-02,2025-12-31,True
5,LIT.L,AU000000LCA6,Litigation Capital Management,0,NaT,NaT,False
6,SVML.L,AU000000SVM6,Sovereign Metals,1020,2021-12-14,2025-12-31,True
7,WNX.L,AU0000162281,Wellnex Life,197,2025-03-21,2025-12-31,True
8,SYN.L,AU0000233538,Synergia Energy,2779,2015-01-02,2025-12-31,True
9,ALL.L,AU0000237554,Atlantic Lithium,2750,2015-02-12,2025-12-31,True


In [4]:
importlib.reload(momentum_data)

missing_symbols = price_coverage.loc[
    ~price_coverage["has_cached_prices"],
    "yf_symbol",
].tolist()

RAW_CACHE_DIR = PROJECT_DIR / "data" / "raw_yahoo_2015_2025_v1"

download_report = momentum_data.download_price_cache(
    missing_symbols,
    RAW_CACHE_DIR,
    start="2015-01-01",
    end="2026-01-01",  # Yahoo's end date is exclusive.
)

display(download_report["status"].value_counts().to_frame("Candidates"))
display(download_report.head(10))

/Users/oscarlewis/Desktop/python/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 450.L"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 4GBL.L"}}}


Processed 10/96 symbols
Processed 20/96 symbols
Processed 30/96 symbols
Processed 40/96 symbols
Processed 50/96 symbols
Processed 60/96 symbols
Processed 70/96 symbols
Processed 80/96 symbols
Processed 90/96 symbols


$VOX.L: possibly delisted; no price data found  (period=5d)


Processed 96/96 symbols


,Candidates
status,
failed,76
downloaded,20


,ticker,status,rows,quote_currency,error,metadata_error
0,450.L,failed,0,None,YFTzMissingError: $450.L: possibly delisted; n...,
1,4GBL.L,failed,0,None,YFTzMissingError: $4GBL.L: possibly delisted; ...,
2,AC8.L,failed,0,None,YFPricesMissingError: $AC8.L: possibly deliste...,
3,AFRN.L,failed,0,None,YFTzMissingError: $AFRN.L: possibly delisted; ...,
4,AGFX.L,failed,0,None,YFTzMissingError: $AGFX.L: possibly delisted; ...,
5,APQ.L,failed,0,None,YFTzMissingError: $APQ.L: possibly delisted; n...,
6,BAY.L,downloaded,1073,GBp,,
7,BBB.L,failed,0,None,YFTzMissingError: $BBB.L: possibly delisted; n...,
8,BELL.L,failed,0,None,YFTzMissingError: $BELL.L: possibly delisted; ...,
9,BIOM.L,failed,0,None,YFTzMissingError: $BIOM.L: possibly delisted; ...,


In [5]:
importlib.reload(momentum_data)

additional_prices = momentum_data.prepare_downloaded_prices(
    missing_symbols,
    RAW_CACHE_DIR,
    schedule,
)

print(
    "Additional tickers:",
    additional_prices.index.get_level_values("ticker").nunique(),
)

display(
    additional_prices.loc[
        additional_prices["observed_row"],
        ["Close", "adj_close", "Volume"],
    ].head(10)
)

Additional tickers: 20


/Users/oscarlewis/Desktop/python/trading_portfolio/uk_portfolio/equity_momentum/src/momentum_data.py:180: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  frame["observed_row"] = frame["observed_row"].fillna(False).astype(bool)
/Users/oscarlewis/Desktop/python/trading_portfolio/uk_portfolio/equity_momentum/src/momentum_data.py:180: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  frame["observed_row"] = frame["observed_row"].fillna(False).astype(bool)
/Users/oscarlewis/Desktop/python/trading_portfolio/uk_portfolio/equity_momentum/src/momentum_data.py:180: Futu

Close  adj_close   Volume
Date       ticker                               
2015-01-02 BLU.L    0.800000   0.800000  26009.0
           BUR.L    1.208750   1.206931      0.0
           CAPD.L   0.227500   0.226171   2908.0
           CIC.L    1.885000   1.884634  33655.0
           DUKE.L   1.056532   1.049550      0.0
           EMVC.L  15.150000  15.150000    194.0
           LIV.L    0.365000   0.361560      0.0
           MAC.L    0.015250   0.015250      0.0
           MAFL.L   0.077500   0.077500   9491.0
           POLR.L   4.085000   4.052203  11983.0

In [6]:
importlib.reload(momentum_data)

additional_flags, additional_audit = momentum_data.audit_price_quality(
    additional_prices
)

display(
    additional_audit
    .sort_values("zero_volume_pct", ascending=False)
    .round(2)
)

display(
    additional_prices.loc[
        additional_flags["suspected_unit_jump"],
        ["Close", "adj_close", "Volume", "Stock Splits"],
    ].head(15)
)

,observed_sessions,missing_price,invalid_price,missing_volume,invalid_volume,zero_volume,ohlc_inconsistent,large_adj_move,suspected_unit_jump,zero_volume_pct
ticker,,,,,,,,,,
VOX.L,801,0,0,0,0,715,75,37,34,89.26
SPDI.L,2779,0,0,0,0,2203,409,7,7,79.27
SCGL.L,2556,0,0,0,0,1491,533,3,0,58.33
LIV.L,2779,0,0,0,0,1608,757,0,0,57.86
BAY.L,1073,0,0,0,0,606,329,4,4,56.48
MAC.L,2779,0,0,0,0,957,1118,4,0,34.44
DFCH.L,1680,0,0,0,0,577,645,0,0,34.35
DUKE.L,2779,0,0,0,0,555,443,0,0,19.97
MAFL.L,2779,0,0,0,0,535,1025,0,0,19.25


,,Close,adj_close,Volume,Stock Splits
Date,ticker,,,,
2024-02-05,VOX.L,0.000035,0.000035,0.0,0.0
2024-02-06,VOX.L,0.003500,0.003500,0.0,0.0
2024-02-19,VOX.L,0.000035,0.000035,0.0,0.0
2024-03-19,VOX.L,0.003500,0.003500,0.0,0.0
2024-04-11,VOX.L,0.000020,0.000020,0.0,0.0
2024-05-07,VOX.L,0.002000,0.002000,0.0,0.0
2024-05-16,VOX.L,0.000020,0.000020,0.0,0.0
2024-07-23,VOX.L,0.002000,0.002000,0.0,0.0
2024-12-09,VOX.L,0.002000,0.002000,0.0,0.0
